# LASSO / Elastic Net Feature Selection (LOOCV)

Finds genes that separate SHAM from C26 mice, **separately for each sex**, using
two penalised logistic regression models:

| Method | `l1_ratio` | Behaviour | Use |
|---|---|---|---|
| **LASSO** | 1.0 | Zeroes out most coefficients; from a group of correlated genes it keeps one and drops the rest | Small headline signature |
| **Elastic Net** | 0.5 | Mixes L1 and L2 penalties; keeps or drops correlated genes together (e.g. Dbp–Ky, r = 0.95) | Larger set for pathway enrichment |

Both are evaluated with leave-one-out cross-validation (LOOCV, 8 folds per sex),
the same setup as `RandomForest_Feature_Importance.ipynb`. This is a cross-check
on `Feature_Weighting.ipynb`, not a replacement for it.

### Steps
1. Load and normalise the data
2. Choose the regularisation strength `C`
3. Sanity check on toy data
4. Run LOOCV
5. Add Hedges' g effect sizes
6. Build the pathway-enrichment gene set (Elastic Net)
7. Save results
8. Compare with attribute weighting
9. Permutation test of LOOCV accuracy
10. External validation of the curated gene sets (GSE276018)

### Limitations
- How many genes get selected depends heavily on `C`. Step 2 sweeps it rather
  than guessing, separately for each method.
- Only 8 samples against thousands of genes.

**Real data:** set `USE_REAL_DATA = True` only for your own local, offline run (see `CLAUDE.md`).

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from lasso_selection import (
    cpm_log_normalize,
    hedges_g,
    permutation_test_accuracy,
    run_loocv_lasso,
    sweep_C,
)

# ---- Run switches ----
USE_REAL_DATA = True         # True only in your own local run, never inside an AI tool session (CLAUDE.md)
RUN_ELASTIC_NET = True       # Elastic Net gives the larger, pathway-enrichment-sized gene set
RUN_PERMUTATION_TEST = True  # Step 9 is slow (~10 min total); set False to skip on reruns

# ---- Data ----
DATA_DIR = "restructured"    # tables, gene lists, saved selection results
FIGURES_DIR = "figures"      # plots/figures, kept separate from data tables
import os
os.makedirs(FIGURES_DIR, exist_ok=True)

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}

# ---- Methods ----
ELASTIC_NET_L1_RATIO = 0.5   # 1.0 = pure LASSO, 0.0 = pure Ridge, in between = Elastic Net

METHODS = ["lasso", "elastic_net"] if RUN_ELASTIC_NET else ["lasso"]
METHOD_LABELS = {"lasso": "LASSO", "elastic_net": "Elastic Net"}
L1_RATIOS = {"lasso": 1.0, "elastic_net": ELASTIC_NET_L1_RATIO}

TOP_N = 10                   # number of top genes printed per table

## Step 1. Load and normalise the data

Checks each sex has 8 samples, the expected number of genes, a 4/4 SHAM/C26
split, and a `Label` column that matches `Group`.

Counts are then CPM + log2 normalised. Raw counts carry each sample's
sequencing depth (library sizes here vary by ~20%), and the per-gene
`StandardScaler` inside each fold does not correct for that. Normalising
changes which genes LASSO picks, so this step matters.

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    # Shape and label checks
    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    data[sex] = {
        "X": cpm_log_normalize(df[gene_cols]),
        "y": df["Label"],
    }
    print(f"{sex}: {df.shape[0]} samples x {len(gene_cols)} genes — checks passed, CPM+log2 normalised")

## Step 2. Choose the regularisation strength `C`

`C` controls how many genes survive:
- **too small** → 0 genes selected (the model just predicts the majority class)
- **too large** → hundreds of genes selected

Each method needs its own sweep, because the same `C` means something different
at a different `l1_ratio`. For each method, look for a `C` with good accuracy and
a sensible `mean_genes_per_fold`:
- **LASSO:** a couple of dozen genes (a minimal signature)
- **Elastic Net:** around a hundred or more (enough for pathway enrichment)

In [ ]:
C_GRID = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.7, 1.0]

for method in METHODS:
    for sex in SEXES:
        print(f"--- {sex} ({METHOD_LABELS[method]}, l1_ratio={L1_RATIOS[method]}) ---")
        print(sweep_C(data[sex]["X"], data[sex]["y"], C_GRID, l1_ratio=L1_RATIOS[method]))
        print()

Set the chosen values from the sweeps above:

In [ ]:
CHOSEN_C = {
    "lasso": 1.0,        # ~25.6 (female) / 37.75 (male) genes per fold — a "couple dozen"
    "elastic_net": 0.3,  # ~104 (female) / 144 (male) genes per fold — sized for pathway enrichment
}

## Step 3. Sanity check on toy data

One clear signal gene (low in SHAM, high in C26) plus four random noise genes.
With the chosen `C`, each method should rank the signal gene first.

In [ ]:
rng = np.random.default_rng(42)

toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})

for method in METHODS:
    toy_summary, toy_accuracy, _ = run_loocv_lasso(
        toy_X, toy_y, C=CHOSEN_C[method], l1_ratio=L1_RATIOS[method]
    )
    print(f"--- {METHOD_LABELS[method]} ---")
    print(toy_summary)
    print(f"toy LOOCV accuracy: {toy_accuracy:.0%}")

    assert toy_summary.index[0] == "Signal", "Signal gene should rank #1 by selection frequency"
    print("PASS — signal gene ranked above all noise genes\n")

## Step 4. Run LOOCV on both sexes

Each fold fits the model on 7 samples and predicts the 1 held out.

`SelectionFreq` = fraction of the 8 folds in which a gene kept a nonzero
coefficient. A gene at 1.0 was picked in every fold.

In [ ]:
summaries = {method: {} for method in METHODS}     # summaries[method][sex] -> per-gene table
fold_details = {method: {} for method in METHODS}  # fold_details[method][sex] -> per-fold predictions

for method in METHODS:
    for sex in SEXES:
        t0 = time.time()
        summary, accuracy, folds = run_loocv_lasso(
            data[sex]["X"], data[sex]["y"], C=CHOSEN_C[method], l1_ratio=L1_RATIOS[method]
        )
        elapsed = time.time() - t0

        summaries[method][sex] = summary
        fold_details[method][sex] = folds

        n_ever_selected = (summary["SelectionFreq"] > 0).sum()

        print(f"--- {sex} ({METHOD_LABELS[method]}) — C={CHOSEN_C[method]}, l1_ratio={L1_RATIOS[method]}, {elapsed:.1f}s ---")
        print(f"LOOCV accuracy:               {accuracy:.0%} ({folds['correct'].sum()}/{len(folds)})")
        print(f"mean genes selected per fold: {folds['n_genes_selected'].mean():.1f}")
        print(f"genes ever selected:          {n_ever_selected}")
        print(f"Top {TOP_N} genes by SelectionFreq:")
        print(summary.head(TOP_N))
        print()

## Step 5. Add Hedges' g effect sizes

Hedges' g is a small-sample-corrected effect size computed directly from the
expression values, independent of either model. It is added as a `HedgesG`
column to every summary table, so a gene's effect size can be compared
between LASSO and Elastic Net.

In [ ]:
for sex in SEXES:
    g = hedges_g(data[sex]["X"], data[sex]["y"]).rename("HedgesG")

    for method in METHODS:
        summaries[method][sex] = summaries[method][sex].join(g)

    top_by_effect = g.reindex(g.abs().sort_values(ascending=False).index).head(TOP_N)
    print(f"--- {sex}: top {TOP_N} genes by |HedgesG| ---")
    print(top_by_effect)
    print()

## Step 6. Build the pathway-enrichment gene set (Elastic Net only)

LASSO's every-fold shortlist is deliberately minimal — good for a headline
signature, too small for pathway enrichment. Elastic Net keeps correlated genes
together, so it is the better source for a larger set.

**Filter:** `SelectionFreq >= 0.25`, i.e. selected in at least 2 of the 8 folds.
This drops one-off inclusions while keeping roughly a hundred or more genes.
`C = 0.3` was chosen after comparing a few values on external-cohort direction
agreement and transfer accuracy.

In [ ]:
PATHWAY_SET_MIN_FREQ = 0.25

pathway_gene_sets = {}

if "elastic_net" in METHODS:
    for sex in SEXES:
        summary = summaries["elastic_net"][sex]
        n_ever_selected = (summary["SelectionFreq"] > 0).sum()

        pathway_set = summary[summary["SelectionFreq"] >= PATHWAY_SET_MIN_FREQ].sort_values(
            ["SelectionFreq", "AbsMeanCoef"], ascending=[False, False]
        )
        pathway_gene_sets[sex] = pathway_set

        print(f"--- {sex} ---")
        print(f"genes ever selected:                     {n_ever_selected}")
        print(f"pathway set (SelectionFreq >= {PATHWAY_SET_MIN_FREQ}):    {len(pathway_set)}")
        print(f"  of which selected in every fold:       {(pathway_set['SelectionFreq'] == 1.0).sum()}")
        print()
else:
    print("RUN_ELASTIC_NET is False — skipping pathway gene set.")

## Step 7. Save results

For each method and sex, saves to `restructured/`:

| File | Contents |
|---|---|
| `{sex}_{method}_selection.csv` | Full per-gene table (`SelectionFreq`, coefficients, `HedgesG`) |
| `{sex}_{method}_shortlist_genes.txt` | Genes selected in every fold |
| `{sex}_{method}_loocv_folds.csv` | Per-fold predictions |
| `{sex}_elastic_net_pathway_geneset.csv` / `.txt` | Step 6 pathway set (table and plain gene list) |

In [ ]:
def write_gene_list(genes, path):
    with open(path, "w") as f:
        f.write("\n".join(genes))


for method in METHODS:
    for sex in SEXES:
        summary = summaries[method][sex]
        folds = fold_details[method][sex]
        shortlist = summary.index[summary["SelectionFreq"] == 1.0].tolist()

        summary_path = f"{DATA_DIR}/{sex}_{method}_selection.csv"
        shortlist_path = f"{DATA_DIR}/{sex}_{method}_shortlist_genes.txt"
        folds_path = f"{DATA_DIR}/{sex}_{method}_loocv_folds.csv"

        summary.to_csv(summary_path, index_label="Gene")
        write_gene_list(shortlist, shortlist_path)
        folds.to_csv(folds_path, index=False)

        print(f"{sex} ({method}):")
        print(f"  {len(summary)}-gene selection table   -> {summary_path}")
        print(f"  {len(shortlist)} genes selected every fold -> {shortlist_path}")
        print(f"  {len(folds)}-fold LOOCV detail          -> {folds_path}")

for sex, pathway_set in pathway_gene_sets.items():
    csv_path = f"{DATA_DIR}/{sex}_elastic_net_pathway_geneset.csv"
    txt_path = f"{DATA_DIR}/{sex}_elastic_net_pathway_geneset.txt"

    pathway_set.to_csv(csv_path, index_label="Gene")
    write_gene_list(pathway_set.index.tolist(), txt_path)

    print(f"{sex} (elastic_net pathway set):")
    print(f"  {len(pathway_set)} genes (table) -> {csv_path}")
    print(f"  plain gene list   -> {txt_path}")

## Step 8. Compare with attribute weighting

Overlap between each method's every-fold shortlist and attribute weighting's
30-gene stability shortlist (`Feature_Weighting.ipynb`, Step 6). Genes that
independent methods agree on are the strongest candidates.

In [ ]:
for sex in SEXES:
    aw_path = f"{DATA_DIR}/{sex}_signature_genes_draft_shortlist.txt"
    try:
        with open(aw_path) as f:
            aw_shortlist = {line.strip() for line in f if line.strip()}
    except FileNotFoundError:
        print(f"{sex}: {aw_path} not found — run Feature_Weighting.ipynb Step 6 first\n")
        continue

    shortlists = {
        method: set(summaries[method][sex].index[summaries[method][sex]["SelectionFreq"] == 1.0])
        for method in METHODS
    }

    print(f"--- {sex} ---")
    print(f"attribute weighting shortlist: {len(aw_shortlist)} genes")
    for method, genes in shortlists.items():
        print(f"{METHOD_LABELS[method]} shortlist: {len(genes)} genes")
        print(f"  ∩ attribute weighting: {sorted(genes & aw_shortlist)}")
    if "elastic_net" in shortlists:
        print(f"LASSO ∩ Elastic Net: {sorted(shortlists['lasso'] & shortlists['elastic_net'])}")
    print()

## Step 9. Permutation test of LOOCV accuracy

With 8 samples split 4/4 there are only C(8,4) = 70 possible balanced label
assignments, so all of them are tested (an exact test, not an approximation).
The full LOOCV pipeline is rerun for each, using the same `C` and `l1_ratio`
chosen above.

**p-value** = fraction of the 70 assignments (including the real one) whose
accuracy is at least as high as the real labels'.

**Runtime:** ~1–2 min per sex for LASSO, ~5–8 min per sex for Elastic Net (its
`saga` solver is slower).

**Note:** with thousands of genes and 7 training samples per fold, even pure
noise can reach high LOOCV accuracy. A non-significant result here is a real
finding to report, not a bug.

In [ ]:
permutation_rows = []

if RUN_PERMUTATION_TEST:
    for method in METHODS:
        for sex in SEXES:
            t0 = time.time()
            observed, null_dist, p = permutation_test_accuracy(
                data[sex]["X"], data[sex]["y"], C=CHOSEN_C[method], l1_ratio=L1_RATIOS[method]
            )
            elapsed = time.time() - t0

            permutation_rows.append({
                "method": method,
                "sex": sex,
                "observed_accuracy": observed,
                "null_mean_accuracy": null_dist.mean(),
                "null_max_accuracy": null_dist.max(),
                "n_permutations": len(null_dist),
                "p_value": p,
            })

            print(f"--- {sex} ({METHOD_LABELS[method]}) — {len(null_dist)} permutations, {elapsed:.1f}s ---")
            print(f"observed: {observed:.0%}   null mean: {null_dist.mean():.0%}   "
                  f"null max: {null_dist.max():.0%}   p = {p:.3f}")
            print()
else:
    print("RUN_PERMUTATION_TEST is False — skipping.")

In [ ]:
if permutation_rows:
    permutation_summary = pd.DataFrame(permutation_rows)
    permutation_summary_path = f"{DATA_DIR}/permutation_test_accuracy_summary.csv"
    permutation_summary.to_csv(permutation_summary_path, index=False)

    print(permutation_summary)
    print(f"\nsaved -> {permutation_summary_path}")

## Step 10. External validation of the curated gene sets (GSE276018)

**Curated gene sets:** hand-picked from the Elastic Net results during review
for the presentation (female: 24 genes, male: 19 genes). They are hardcoded
below, not re-derived here.

For each sex, one Elastic Net model is fitted on all 8 of our own samples using
only the curated genes, then tested on an independent cohort (GSE276018) that
was never used during discovery. Two checks:

- **Transfer accuracy** — does the model correctly classify the external mice?
  Fragile: it needs the right decision boundary, not just the right direction.
- **Direction agreement** — does each gene's coefficient sign match the sign of
  its Hedges' g in the external cohort? A softer, more robust check.

Requires `GSE276018_precachectic_subset_with_symbols.csv` and
`GSE276018_precachectic_sample_metadata.csv` in the project root (the same files
`External_Dataset_Pooling.ipynb` uses; gitignored, local only).

In [ ]:
CURATED_GENE_SETS = {
    "female": [
        "Prodh", "Per3", "Cd163", "Ppp1r3c", "Disp1", "Hlf", "Dpy19l3", "Itpripl1",
        "Stard3", "Crim1", "Per2", "Ank2", "Grrp1", "Uevld", "Large1", "Nrp2",
        "Actr3", "Rilpl1", "Cdkn1a", "Ppargc1b", "Angpt1", "Cldn15", "Tnfrsf1b", "Dbp",
    ],
    "male": [
        "Eif2a", "Tmem86a", "Slc25a10", "Anapc16", "Grrp1", "Ppp1r3c", "Eno1",
        "Cdkn1a", "Rorc", "Dbp", "Erfe", "Acbd4", "Tmem176b", "Tab1", "Tagap",
        "Ddx59", "Abcb9", "Dcakd", "Dok5",
    ],
}

EXTERNAL_EXPR_PATH = "GSE276018_precachectic_subset_with_symbols.csv"
EXTERNAL_METADATA_PATH = "GSE276018_precachectic_sample_metadata.csv"

In [ ]:
ext_expr = pd.read_csv(EXTERNAL_EXPR_PATH, index_col="EnsemblID")
ext_meta = pd.read_csv(EXTERNAL_METADATA_PATH)
ext_all_genes = sorted(set(ext_expr["Symbol"].dropna()))


def load_external_cohort(sex):
    # Returns (CPM+log2 normalised samples x genes, 0/1 labels) for one sex.
    sex_meta = ext_meta[ext_meta["sex"] == sex]
    y = pd.Series((sex_meta["Group"] == "C26_10day").astype(int).values)

    # Normalise on the FULL external gene panel before subsetting — the CPM
    # library-size denominator must be the whole panel, not just the curated genes.
    wide = ext_expr.set_index("Symbol")[sex_meta["title"].tolist()].T
    wide = wide.loc[:, ~wide.columns.duplicated()]
    wide = wide[[g for g in ext_all_genes if g in wide.columns]].astype(float)
    X = cpm_log_normalize(wide).reset_index(drop=True)
    return X, y


def zscore_within_study(X):
    # Each study is standardised on its own — batch effects break the usual
    # "fit the scaler on train, apply it to test" assumption across studies.
    return (X - X.mean()) / X.std(ddof=0).replace(0, 1)

In [ ]:
external_validation_results = {}

for sex in SEXES:
    X_ext, y_ext = load_external_cohort(sex)
    genes = [g for g in CURATED_GENE_SETS[sex] if g in X_ext.columns]

    X_train = zscore_within_study(data[sex]["X"][genes])
    X_test = zscore_within_study(X_ext[genes])

    # No explicit `penalty=` — matches lasso_selection.py. Newer sklearn (>=1.8)
    # deprecates penalty="elasticnet" in favour of l1_ratio alone, and mixing the
    # two styles gave different coefficients depending on sklearn version.
    clf = LogisticRegression(
        l1_ratio=ELASTIC_NET_L1_RATIO, C=CHOSEN_C["elastic_net"],
        solver="saga", max_iter=5000, random_state=51,
    )
    clf.fit(X_train.values, data[sex]["y"])

    # Check 1: transfer accuracy
    accuracy = (clf.predict(X_test.values) == y_ext.values).mean()
    baseline = max(y_ext.mean(), 1 - y_ext.mean())

    # Check 2: direction agreement (genes the L1 penalty dropped are excluded)
    coefs = pd.Series(clf.coef_[0], index=genes)
    nonzero = coefs[coefs != 0]
    ext_g = hedges_g(X_ext[nonzero.index], y_ext)
    agree = np.sign(nonzero) == np.sign(ext_g)

    external_validation_results[sex] = {
        "accuracy": accuracy,
        "baseline": baseline,
        "n_genes_used": len(nonzero),
        "n_genes_total": len(genes),
        "direction_agree": agree.sum(),
        "direction_total": len(agree),
    }

    print(f"--- {sex} ---")
    print(f"curated genes found in external data: {len(genes)}/{len(CURATED_GENE_SETS[sex])}")
    print(f"genes with nonzero coefficient:       {len(nonzero)}/{len(genes)}")
    print(f"transfer accuracy:   {accuracy:.0%} (majority baseline {baseline:.0%}, n={len(y_ext)})")
    print(f"direction agreement: {agree.sum()}/{len(agree)} ({agree.mean():.0%})")
    print()

In [ ]:
external_validation_summary = pd.DataFrame(external_validation_results).T
external_validation_summary.index.name = "sex"

external_validation_path = f"{DATA_DIR}/curated_geneset_external_validation.csv"
external_validation_summary.to_csv(external_validation_path)

print(external_validation_summary)
print(f"\nsaved -> {external_validation_path}")

## Step 11. True vs. predicted plot (presentation figure)

Refits the same curated-gene Elastic Net model per sex and plots true group
vs. predicted group side by side for every external mouse. Samples are
relabelled to match the project's own naming convention (`FSHAM`/`FC26` for
female, `MSHAM`/`MC26` for male) instead of the raw GEO sample titles.

Two dots per row (true, predicted), coloured by group; a dashed connector
flags a misclassified sample, a solid connector flags a correct one — no
probability values shown, just the categorical call, kept deliberately
minimal (high data-ink ratio, no gridlines/borders, one shared legend).

In [ ]:
import matplotlib.pyplot as plt

GROUP_LABELS = {0: "SHAM", 1: "C26"}
MUTED = "#52514e"
RED = "#e34948"
SEX_PREFIX = {"female": "F", "male": "M"}

true_vs_pred = {}

for sex in SEXES:
    X_ext, y_ext = load_external_cohort(sex)
    genes = [g for g in CURATED_GENE_SETS[sex] if g in X_ext.columns]
    X_train = zscore_within_study(data[sex]["X"][genes])
    X_test = zscore_within_study(X_ext[genes])

    clf = LogisticRegression(
        l1_ratio=ELASTIC_NET_L1_RATIO, C=CHOSEN_C["elastic_net"],
        solver="saga", max_iter=5000, random_state=51,
    )
    clf.fit(X_train.values, data[sex]["y"])
    preds = clf.predict(X_test.values)

    prefix = SEX_PREFIX[sex]
    n_sham = (y_ext == 0).sum()
    n_c26 = (y_ext == 1).sum()
    labels = [f"{prefix}SHAM{i}" for i in range(1, n_sham + 1)] + [f"{prefix}C26{i}" for i in range(1, n_c26 + 1)]

    true_vs_pred[sex] = pd.DataFrame({"label": labels, "true": y_ext.values, "pred": preds})


def plot_panel(ax, df, title):
    n = len(df)
    ax.set_xlim(0, 3.2)
    ax.set_ylim(-1, n)
    ax.invert_yaxis()

    ax.text(0, -0.8, "Sample", fontsize=10, fontweight="bold", ha="left", color=MUTED)
    ax.text(1.7, -0.8, "True", fontsize=10, fontweight="bold", ha="center", color=MUTED)
    ax.text(2.7, -0.8, "Pred", fontsize=10, fontweight="bold", ha="center", color=MUTED)
    ax.plot([0, 3.2], [-0.35, -0.35], color="#d8d7d2", lw=1)

    for i, row in df.iterrows():
        mismatch = row["true"] != row["pred"]
        text_color = RED if mismatch else "#0b0b0b"
        if mismatch:
            ax.add_patch(plt.Rectangle((0, i - 0.45), 3.2, 0.9, color="#fbeceb", zorder=0))
        ax.text(0, i, row["label"], fontsize=9.5, ha="left", va="center", color="#0b0b0b", zorder=2)
        ax.text(1.7, i, GROUP_LABELS[row["true"]], fontsize=9.5, ha="center", va="center", color="#0b0b0b", zorder=2)
        ax.text(2.7, i, GROUP_LABELS[row["pred"]], fontsize=9.5,
                 fontweight="bold" if mismatch else "normal", ha="center", va="center", color=text_color, zorder=2)

    ax.set_title(title, fontsize=11, loc="left", fontweight="bold", pad=14)
    ax.axis("off")


fig, axes = plt.subplots(1, 2, figsize=(7.5, 5.8))
plot_panel(axes[0], true_vs_pred["female"], f"Female (n={len(true_vs_pred['female'])})")
plot_panel(axes[1], true_vs_pred["male"], f"Male (n={len(true_vs_pred['male'])})")
fig.suptitle("External validation: true vs. predicted group", fontsize=12, y=0.99)
fig.tight_layout()

true_vs_pred_path = f"{FIGURES_DIR}/true_vs_pred.png"
fig.savefig(true_vs_pred_path, dpi=200, bbox_inches="tight", facecolor="white")
print(f"saved -> {true_vs_pred_path}")